In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E4B-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = False, # Changed from "unsloth" to False to fix the runtime error
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/363k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 32,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 32,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

In [ ]:
!unzip dataset_coa_basic_Slides.zip -d /content

Archive:  dataset_coa_basic_Slides.zip
   creating: /content/dataset_coa_basic_Slides/
  inflating: /content/dataset_coa_basic_Slides/all_lectures.csv  
  inflating: /content/dataset_coa_basic_Slides/all_lectures_analysis.csv  
   creating: /content/dataset_coa_basic_Slides/per_lecture/
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_1.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_10.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_11.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_12.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_13.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_14.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_15.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_16.csv  
  inflating: /content/dataset_coa_basic_Slides/per_lecture/lecture_17.csv  
  inflating: /content/dataset

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('/content/all_lectures.csv.xls')

In [ ]:
df.columns

Index(['lecture_number', 'slide_set', 'transcript_merged'], dtype='object')

In [ ]:
df['slide_set'][0]

"['/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide1.png', '/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide3.png', '/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide4.png', '/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide5.png', '/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide6.png', '/home/proy/edu/coa_basic_Slides/slides/lec1-2/lec1-2_slide7.png']"

In [ ]:
print(type(df["slide_set"][0]))

<class 'str'>


In [ ]:
import ast

df["slide_set"] = df["slide_set"].apply(ast.literal_eval)

In [ ]:
print(type(df["slide_set"][0]))

<class 'list'>


In [ ]:
import os
import re

OLD_PREFIX = "/home/proy/edu/coa_basic_Slides"
NEW_PREFIX = "/content/dataset_coa_basic_Slides"

def convert_single_path(path):
    path = path.replace(OLD_PREFIX, NEW_PREFIX)

    filename = os.path.basename(path)

    match = re.match(r"(.*)_slide(\d+)\.png", filename)

    if match:
        prefix = match.group(1)
        slide_num = int(match.group(2))
        filename = f"{prefix}_p{slide_num:04d}.png"

    return os.path.join(os.path.dirname(path), filename)

for idx in df.index:
    slide_paths = []
    for path in df.at[idx, "slide_set"]:
        slide_paths.append(convert_single_path(path))
    df.at[idx, "slide_set"] = slide_paths

In [ ]:
print(df["slide_set"][0])

['/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0001.png', '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0003.png', '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0004.png', '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0005.png', '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0006.png', '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0007.png']


In [ ]:
for p in df["slide_set"][0]:
    print(p, os.path.exists(p))

/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0001.png True
/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0003.png True
/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0004.png True
/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0005.png True
/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0006.png True
/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0007.png True


In [ ]:
df.columns

Index(['lecture_number', 'slide_set', 'transcript_merged'], dtype='object')

In [ ]:
df["slide_set"][0]

['/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0001.png',
 '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0003.png',
 '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0004.png',
 '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0005.png',
 '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0006.png',
 '/content/dataset_coa_basic_Slides/slides/lec1-2/lec1-2_p0007.png']

In [ ]:
from PIL import Image
from datasets import Dataset

instruction = (
    "Generate a detailed lecture narrative by understanding all the slides."
)

def convert_to_conversation(row):

    content = []

    for img_path in row["slide_set"]:
        content.append(
            {
                "type": "image",
                "image": Image.open(img_path).convert("RGB"),
            }
        )

    content.append(
        {
            "type": "text",
            "text": instruction,
        }
    )

    transcript = row["transcript_merged"].replace("[NEXT_SLIDE]", "\n\n")

    return {
        "messages": [
            {
                "role": "user",
                "content": content,
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": transcript,
                    }
                ],
            },
        ]
    }

dataset = Dataset.from_list(
    [convert_to_conversation(row) for _, row in df.iterrows()]
)

In [ ]:
converted_dataset = []

for _, row in df.iterrows():
    converted_dataset.append(convert_to_conversation(row))

In [ ]:
type(converted_dataset)
type(converted_dataset[0]["messages"][0]["content"][0]["image"])

PIL.Image.Image

In [ ]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=601x451>},
    {'type': 'text',
     'text': 'Generate a detailed lecture narrative by understanding all the slides.'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'If you see a computer design in a layered view, there are several layers starting from the\napplication layer and all the way up to transistor layer. And each of these layers specifies set\nof operations and for example, if you consider application layer. So, talks about the type of\napplications 

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(model, processor)

bad = []

for i in range(len(converted_dataset)):
    try:
        collator([converted_dataset[i]])
    except Exception as e:
        bad.append((i, str(e)))

print("Bad samples:", len(bad))
print(bad[:10])

Unsloth: Model does not have a default image size - using 512
Bad samples: 9
[(1, "Mismatch in `image` token count between text and `input_ids`. Got ids=[2029] and text=[2394]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`."), (6, "Mismatch in `image` token count between text and `input_ids`. Got ids=[2029] and text=[2128]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`."), (8, "Mismatch in `image` token count between text and `input_ids`. Got ids=[2029] and text=[2660]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`."), (25, "Mismatch in `image` token count between text and `input_ids`. Got ids=[2029] and text=[2660]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`."), (26, "Mismatch in `image` token count between text and `input_ids`. Got ids=[2029] and text=[2128]. Likely due to `truncation='max_length'`. 

In [ ]:
bad_indices = {1, 6, 8, 25, 26, 27, 30, 31, 32}

filtered_dataset = [
    sample
    for i, sample in enumerate(converted_dataset)
    if i not in bad_indices
]

print(f"Original dataset : {len(converted_dataset)}")
print(f"Filtered dataset : {len(filtered_dataset)}")

Original dataset : 33
Filtered dataset : 24


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=filtered_dataset,
    processing_class=processor,          # or processor.tokenizer if processor doesn't work
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8, # Increased to 8
        learning_rate=2e-4,
        max_steps=10,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=4096, # Reduced from 16384 to 4096
    ),
)

Unsloth: Model does not have a default image size - using 512


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 4 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 82,444,288 of 8,078,600,736 (1.02% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with gu

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.50 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.44 GiB is free. Including non-PyTorch memory, this process has 12.12 GiB memory in use. Of the allocated memory 11.95 GiB is allocated by PyTorch, and 40.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)